> “你做一个Agent产品，需要把模型、跟工具、和 Context 结合起来。”
- pe vs. ce
    - pe: craft of wording the instruction itself.
        - role assignment, few-shot examples, CoT prompting, constratin setting
    - ce: system level, 编排层
- context engineering means context management
- key points
    - 从数据的角度：
        - sufficient context，提供的信息是否是足够的；
        - 提供的信息是否是冗余的；
    - 从字面意思来说，避免 context 超出模型的 context window
        - system prompt + messsages(持续变长) + tools + rag + reasoning
        - 尤其是 AI agent 的范式下，相当长的工具调用列表；
        - 模型能力会随着上下文变长而能力衰减；
            - lost in the middle：只记得开头和结尾，倒U型曲线（横轴正确答案的位置，纵轴准确率）；
    - 通俗理解
        - 需要的放进去，不需要的拿出来；
        - special tricks：dynamic update plan（todo list） 
- examples
    - https://github.com/FareedKhan-dev/contextual-engineering-guide/blob/main/contextual_engineering.ipynb
    - https://github.com/langchain-ai/how_to_fix_your_context

### 类比计算机

- LLM: CPU
    - 计算、推理与生成，output = f(input)
    - 每次调用都是无状态的，不存储信息
- 上下文窗口：RAM（working memory）
- 上下文工程（context engineering）：OS
    - 调度，编排
    - 内存管理器，负责从`硬盘`（外部文件）向 `RAM`高效调度关键信息

### system prompt vs. user prompt 

- system prompt: 稳定，全局的，整体的 instructions
    - https://cookbook.openai.com/examples/gpt4-1_prompting_guide
    - The Butterfly Effect of Altering Prompts: How Small Changes and Jailbreaks Affect Large Language Model Performance
- 产品级 system prompt 参考：https://docs.claude.com/en/release-notes/system-prompts
    - xml tag format 整体的 system prompts: 可读 & 可维护, 可程序化/可快速替换，https://docs.claude.com/en/docs/build-with-claude/prompt-engineering/use-xml-tags
        - `<behavior_instructions></behavior_instructions>`: 总纲
        - `<general_claude_info></general_claude_info>`: 产品/模型元信息与可对外说的知识边界
        - `<refusal_handling></refusal_handling>`: 拒绝与安全策略的触发条件与拒绝方式（涉及恶意代码、伤害信息、未成年人等）。
        - `<tone_and_formatting></tone_and_formatting>`: 语气与格式规范（什么时候用段落、是否允许列表、是否用 emoji 等）。
        - `<user_wellbeing></user_wellbeing>`: 福祉/心理类守则；当检测到高风险信号时的对话策略。
        - `<knowledge_cutoff>` 与内部的 `<election_info>`：时间边界与硬编码事实块；告诉模型“哪些事实可以直接断言，哪些必须搜索”。
    - 回应风格与格式：`Claude never starts its response by saying a question or idea or observation was good, great, fascinating, profound, excellent, or any other positive adjective. It skips the flattery and responds directly.`
    - 错误处理与互动细节：`If the user corrects Claude or tells Claude it’s made a mistake, then Claude first thinks through the issue carefully before acknowledging the user, since users sometimes make errors themselves.`

### in-context learning

- Context 中提供 query 具体的情景（上下文），llm更好地grounded（retrieval）具体针对性的回答
- icl，给具体例子，给的例子一定要更general，避免模型overfitting到给定的 example 里，降低其泛化性和推理能力；
    - gemini 1.5 系列论文展示的小语种语言的例子其实就是通过示例来 learning 的；而不是语法？

## context engineering (langchain)

- https://blog.langchain.com/context-engineering-for-agents/
- Context engineering is **becoming a craft** that agents builders should aim to master. Here, we covered a few common patterns seen across many popular agents today:
    - Writing context - saving it outside the context window to help an agent perform a task.
    - Selecting context - pulling it into the context window to help an agent perform a task.
    - Compressing context - retaining only the tokens required to perform a task.
    - Isolating context - splitting it up to help an agent perform a task.
        - Mulit-Agent，Separation of Concern

### compress context

- A few places where summarization can be applied
    - summarize message history
    - summarize tool feedback
        - agent-agent boundaries to reduce tokens during knowledge hand-off. 
- gemini-cli
    - https://github.com/google-gemini/gemini-cli/blob/main/packages/core/src/core/prompts.ts#L305-L361
- claude-code
    - Claude Code runs “auto-compact” after you exceed **95%** of the context window and it will summarize the full trajectory of user-agent interactions.
        - https://docs.anthropic.com/en/docs/claude-code/costs
```
# Summary instructions

When you are using compact, please focus on test output and code changes
```